In [167]:
# we always start with a dataset to train on. Install tiny shakespeare dataset
import urllib.request
import torch

url = "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x31ca489f0>)

In [168]:
batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

with open("input.txt", "r") as f:
    text = f.read()

In [169]:
print("length of the dataset in characters: ", len(text))

length of the dataset in characters:  1115394


In [170]:
#print first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [171]:
# we get all the characters occuring in the text
chars = sorted(list(set(text)))
# unique set of characters in sorted orders

vocab_size = len(chars)
print(''.join(chars))
print("vocabulary size:", vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocabulary size: 65


In [172]:
# now we need to tokenize the entire text
# aslinda her bir karakteri bir integer ile temsil edecegiz
# chracter level tokenizer kullaniyoruz
# ama daha profesyonel tokenizerlar mevcut ornegin her bir kelimeyi bir string ile 
# ifade edebilen tokenizerler var 
# REPOYA BAK -> openai/tiktoken

stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]     # take a string output a list on integers
decode = lambda l: ''.join([itos[i] for i in l])    # take list of integers output a string

print(encode("a hii there"))
print(decode(encode("a hii there")))

[39, 1, 46, 47, 47, 1, 58, 46, 43, 56, 43]
a hii there


In [173]:
# we will encode (tokenize) the entire training set and store it in torch.Tensor

import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])      # translation of first 1000 charactes of the text

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [174]:
# we need to train the machine 
# split the data into train and validation sets

n = int(0.8 * len(data))    # 80% of the data for train
train_data = data[:n]
test_data = data[n:]

In [175]:
block_size = 8  # egitim yapacagim block size'im
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [176]:
x = train_data[:block_size]
y = train_data[1 : block_size + 1]

# block icerisindeki her bir token icin ondan sonra gelecek olan tokenlari estimate etmeye calisiyoruz

for t in range(block_size):
    context = x[:t+1]   # t. token'a kadar olanlar elimdekiler (context)
    target = y[t]       # (t+1). token'a gelmesi muhtemel olan tokeni istiyorum
    print(f"when input is {context} the target is {target}")

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [177]:
torch.manual_seed(1337)
batch_size = 4      # how many independent sequences will we process in parallel
block_size = 8      # what is the maximum length for the prediction

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))      # we are generating batch size number of random offsets in the dataset
    # dataset icerisinden offsett uzunlugunda rastgele kisimlar aliyoruz ve onlar uzerinden tahmin yapiyoruz

    x = torch.stack([data[i : i+block_size] for i in ix])
    y = torch.stack([data[i+1 : i + block_size+1] for i in ix])

    # bu for donguleri ile her bir block size boyutundaki satirlari matrix haline getiriyoruz

    return x,y

xb, yb = get_batch('train')
print('inputs: ')
print(xb.shape)
print(xb)
print('')
print('targets: ')
print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs: 
torch.Size([4, 8])
tensor([[58, 63,  8,  0,  0, 19, 24, 27],
        [39, 59, 45, 46, 58,  1, 46, 43],
        [49, 43, 57,  1, 53, 50, 42,  1],
        [52, 41, 47, 43, 52, 58,  1, 56]])

targets: 
torch.Size([4, 8])
tensor([[63,  8,  0,  0, 19, 24, 27, 33],
        [59, 45, 46, 58,  1, 46, 43,  1],
        [43, 57,  1, 53, 50, 42,  1, 46],
        [41, 47, 43, 52, 58,  1, 56, 47]])
when input is [58] the target: 63
when input is [58, 63] the target: 8
when input is [58, 63, 8] the target: 0
when input is [58, 63, 8, 0] the target: 0
when input is [58, 63, 8, 0, 0] the target: 19
when input is [58, 63, 8, 0, 0, 19] the target: 24
when input is [58, 63, 8, 0, 0, 19, 24] the target: 27
when input is [58, 63, 8, 0, 0, 19, 24, 27] the target: 33
when input is [39] the target: 59
when input is [39, 59] the target: 45
when input is [39, 59, 45] the target: 46
when input is [39, 59, 45, 46] the target: 58
when input is [39, 59, 45, 46, 58] the target: 1
when input is [39, 59, 45, 46

In [178]:
print(xb)

tensor([[58, 63,  8,  0,  0, 19, 24, 27],
        [39, 59, 45, 46, 58,  1, 46, 43],
        [49, 43, 57,  1, 53, 50, 42,  1],
        [52, 41, 47, 43, 52, 58,  1, 56]])


In [ ]:
import torch 
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# burda yaptigimiz sey her bir tokendan sonra hangi tokenin gelebilecegine dair skorlar olusturmak
# ayni anda 4 farkli sequence (satir) isleniyor
# her bir sequence icin 8 farkli token konumu mevcut
# her bir konumda bir sonraki token icin 65 adayin her birine ait skor uretiliyor

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)   # B T C 
        pos_embd = self.position_embedding_table(torch.arange(T, device=idx.device))

        x = tok_emb + pos_embd
        logits = self.lm_head(x)  # B T vocab_size
        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)     # logitler ve gercek degerler arasindaki farka bakar
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context

        for _ in range(max_new_tokens):
            # get predictions
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)

            # focus only on last time step
            logits = logits[:, -1, :] # becomes (B,C)

            # get probabilities
            probs = F.softmax(logits, dim=-1)

            # sample from the distributon
            idx_next = torch.multinomial(probs, num_samples=1)

            # append the sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx


model = BigramLanguageModel(vocab_size)

logits, loss = model(xb,yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1,1), dtype=torch.long)  # batch=1, time=1 1x1 tensor holding ones
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.3316, grad_fn=<NllLossBackward0>)


IndexError: index out of range in self

In [ ]:
# pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
batch_size = 32

for iter in range(max_iters):
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaulate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.791837692260742


In [ ]:
idx = torch.zeros((1,1), dtype=torch.long)  # batch=1, time=1 1x1 tensor holding ones
print(decode(model.generate(idx, max_new_tokens=400)[0].tolist()))


jxrdfof he
E:
A!q,
Wetti
IorbengamTqtofaljupY: f miPAM:Tstovo.-der wes, cthat
DWereEX3'd, sr nxEn Qog-ggOEZXtonenourFonsd, ntn CArensCEEwo t.
N:
KtKININERImy,,SZRlgouduskes k,
sBedUElig-thiy mowiKks OUrno, yAPOL:we te necrisBEm e ndivesd
LS.
n 'QnqAn, d nkStet,, r isiceQqwin ustisgIORENze, tic-dh $GhecP, W?YNVblevirve?ngCIVjNen,lCOLttttAJ&?qVXSBkskandYowoTICOr $ foCKowxithow aungb
gmndelin3QNH&f w


In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
xbow = torch.zeros((B,T,C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x

torch.allclose(xbow, xbow2)

True

In [ ]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b

print('a=')
print(a)
print()
print('b=')
print(b)
print()
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])

c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
